# Week 4 — Baseline Models and Evaluation

## 1. Chronological Train-Test Split — Deepanshu

This notebook prepares the preprocessed Delhi air-quality dataset for model development. The data is split chronologically so that earlier observations are used for training and later observations are used for testing.

A chronological split reflects the real-world forecasting task: using historical observations to predict future air quality. A random split is avoided because it can mix earlier and later observations and produce an unrealistic evaluation of future forecasting performance.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("../data/processed/delhi_preprocessed.csv")

processed_df = pd.read_csv(
    data_path,
    index_col="Date",
    parse_dates=True
)

processed_df = processed_df.sort_index()

print("Dataset shape:", processed_df.shape)
print("First date:", processed_df.index.min())
print("Last date:", processed_df.index.max())
print("\nAvailable columns:")
print(processed_df.columns.tolist())

Dataset shape: (2009, 14)
First date: 2015-01-01 00:00:00
Last date: 2020-07-01 00:00:00

Available columns:
['City', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'AQI', 'AQI_Bucket']


In [2]:
print("Date index:", processed_df.index.name)
print("Duplicate dates:", processed_df.index.duplicated().sum())
print("Dates sorted:", processed_df.index.is_monotonic_increasing)

print("\nMissing values in key columns:")
for col in ["PM2.5", "AQI", "AQI_Bucket"]:
    print(f"{col}: {processed_df[col].isna().sum()}")

Date index: Date
Duplicate dates: 0
Dates sorted: True

Missing values in key columns:
PM2.5: 2
AQI: 10
AQI_Bucket: 10


In [3]:
# Create next-day targets in memory; do not modify the saved CSV
processed_df["PM2.5_tomorrow"] = processed_df["PM2.5"].shift(-1)
processed_df["AQI_Bucket_tomorrow"] = processed_df["AQI_Bucket"].shift(-1)

# Keep only rows with an observed target for each task
regression_df = processed_df.dropna(
    subset=["PM2.5_tomorrow"]
).copy()

classification_df = processed_df.dropna(
    subset=["AQI_Bucket_tomorrow"]
).copy()

# Split according to the date being predicted, not just the feature date
split_date = pd.Timestamp("2020-01-01")

regression_train = regression_df.loc[
    (regression_df.index + pd.Timedelta(days=1)) < split_date
].copy()

regression_test = regression_df.loc[
    (regression_df.index + pd.Timedelta(days=1)) >= split_date
].copy()

classification_train = classification_df.loc[
    (classification_df.index + pd.Timedelta(days=1)) < split_date
].copy()

classification_test = classification_df.loc[
    (classification_df.index + pd.Timedelta(days=1)) >= split_date
].copy()

print("REGRESSION — next-day PM2.5")
print("Train rows:", len(regression_train))
print("Test rows:", len(regression_test))
print("Train feature dates:",
      regression_train.index.min(), "to", regression_train.index.max())
print("Test feature dates:",
      regression_test.index.min(), "to", regression_test.index.max())

print("\nCLASSIFICATION — next-day AQI category")
print("Train rows:", len(classification_train))
print("Test rows:", len(classification_test))

# Confirm no overlap in the dates being predicted
assert (
    regression_train.index.max() + pd.Timedelta(days=1)
) < split_date

assert (
    regression_test.index.min() + pd.Timedelta(days=1)
) >= split_date

assert (
    classification_train.index.max() + pd.Timedelta(days=1)
) < split_date

assert (
    classification_test.index.min() + pd.Timedelta(days=1)
) >= split_date

print("\nPASS: chronological split verified.")

REGRESSION — next-day PM2.5
Train rows: 1823
Test rows: 183
Train feature dates: 2015-01-01 00:00:00 to 2019-12-30 00:00:00
Test feature dates: 2019-12-31 00:00:00 to 2020-06-30 00:00:00

CLASSIFICATION — next-day AQI category
Train rows: 1815
Test rows: 183

PASS: chronological split verified.


### 2. Prepare Features for Baseline Models

The dataset is divided chronologically, using observations before 2020 for training and observations from 2020 onward for testing. Two prediction tasks are considered: regression to predict next-day PM2.5 concentration and classification to predict the next-day AQI category.

Target columns and non-predictive columns are excluded from the input features to prevent target leakage. Missing feature values will be handled separately based on the preprocessing strategy.

In [4]:
# Define target columns
regression_target = "PM2.5_tomorrow"
classification_target = "AQI_Bucket_tomorrow"

# Columns that must not be used as predictors
excluded_columns = [
    "City",
    "AQI_Bucket",
    "PM2.5_tomorrow",
    "AQI_Bucket_tomorrow"
]

# Use only numeric pollutant/AQI features
feature_columns = [
    col for col in processed_df.select_dtypes(include="number").columns
    if col not in excluded_columns
]

print("Predictor columns:")
print(feature_columns)

print("\nNumber of predictors:", len(feature_columns))

# Build train/test feature matrices
X_train_reg = regression_train[feature_columns]
y_train_reg = regression_train[regression_target]

X_test_reg = regression_test[feature_columns]
y_test_reg = regression_test[regression_target]

X_train_clf = classification_train[feature_columns]
y_train_clf = classification_train[classification_target]

X_test_clf = classification_test[feature_columns]
y_test_clf = classification_test[classification_target]

print("\nRegression feature shapes:")
print("X_train:", X_train_reg.shape)
print("X_test:", X_test_reg.shape)

print("\nClassification feature shapes:")
print("X_train:", X_train_clf.shape)
print("X_test:", X_test_clf.shape)

print("\nMissing feature values in regression training data:")
print(X_train_reg.isna().sum().sum())

print("\nMissing feature values in classification training data:")
print(X_train_clf.isna().sum().sum())

Predictor columns:
['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'AQI']

Number of predictors: 12

Regression feature shapes:
X_train: (1823, 12)
X_test: (183, 12)

Classification feature shapes:
X_train: (1815, 12)
X_test: (183, 12)

Missing feature values in regression training data:
263

Missing feature values in classification training data:
239


In [5]:
# Inspect missing values in the training features
missing_summary = pd.DataFrame({
    "Missing Count": X_train_reg.isna().sum(),
    "Missing Percentage": (
        X_train_reg.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "Missing Count",
    ascending=False
)

display(missing_summary[missing_summary["Missing Count"] > 0])

print("Regression training rows:", len(X_train_reg))
print("Rows with any missing features:", X_train_reg.isna().any(axis=1).sum())

print(
    "Classification rows with any missing features:",
    X_train_clf.isna().any(axis=1).sum()
)

,Missing Count,Missing Percentage
SO2,107,5.87
O3,79,4.33
PM10,66,3.62
AQI,10,0.55
PM2.5,1,0.05


Regression training rows: 1823
Rows with any missing features: 119
Classification rows with any missing features: 112


In [6]:
from sklearn.impute import SimpleImputer

# Create separate imputers for regression and classification
reg_imputer = SimpleImputer(strategy="median")
clf_imputer = SimpleImputer(strategy="median")

# Fit only on training data, then transform both sets
X_train_reg_imputed = pd.DataFrame(
    reg_imputer.fit_transform(X_train_reg),
    columns=X_train_reg.columns,
    index=X_train_reg.index
)

X_test_reg_imputed = pd.DataFrame(
    reg_imputer.transform(X_test_reg),
    columns=X_test_reg.columns,
    index=X_test_reg.index
)

X_train_clf_imputed = pd.DataFrame(
    clf_imputer.fit_transform(X_train_clf),
    columns=X_train_clf.columns,
    index=X_train_clf.index
)

X_test_clf_imputed = pd.DataFrame(
    clf_imputer.transform(X_test_clf),
    columns=X_test_clf.columns,
    index=X_test_clf.index
)

# Verify that no missing feature values remain
print("Regression training missing values:",
      X_train_reg_imputed.isna().sum().sum())

print("Regression test missing values:",
      X_test_reg_imputed.isna().sum().sum())

print("Classification training missing values:",
      X_train_clf_imputed.isna().sum().sum())

print("Classification test missing values:",
      X_test_clf_imputed.isna().sum().sum())

print("\nFeature preparation complete.")

Regression training missing values: 0
Regression test missing values: 0
Classification training missing values: 0
Classification test missing values: 0

Feature preparation complete.


### 3. Baseline Models and Evaluation

Train baseline regression and classification models using the chronologically split data. Evaluate next-day PM2.5 prediction with regression metrics and next-day AQI category prediction with classification metrics.

In [7]:
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report
)
import numpy as np

# Baseline regression: predict the training target's mean
regression_baseline = DummyRegressor(strategy="mean")
regression_baseline.fit(X_train_reg_imputed, y_train_reg)

reg_predictions = regression_baseline.predict(X_test_reg_imputed)

print("REGRESSION BASELINE — Next-day PM2.5")
print("MAE:", round(mean_absolute_error(y_test_reg, reg_predictions), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test_reg, reg_predictions)), 3))
print("R²:", round(r2_score(y_test_reg, reg_predictions), 3))

# Baseline classification: predict the most frequent training category
classification_baseline = DummyClassifier(strategy="most_frequent")
classification_baseline.fit(X_train_clf_imputed, y_train_clf)

clf_predictions = classification_baseline.predict(X_test_clf_imputed)

print("\nCLASSIFICATION BASELINE — Next-day AQI category")
print("Accuracy:", round(accuracy_score(y_test_clf, clf_predictions), 3))
print("\nClassification report:")
print(classification_report(
    y_test_clf,
    clf_predictions,
    zero_division=0
))

REGRESSION BASELINE — Next-day PM2.5
MAE: 60.77
RMSE: 68.89
R²: -0.527

CLASSIFICATION BASELINE — Next-day AQI category
Accuracy: 0.191

Classification report:
              precision    recall  f1-score   support

    Moderate       0.00      0.00      0.00        98
        Poor       0.19      1.00      0.32        35
Satisfactory       0.00      0.00      0.00        25
      Severe       0.00      0.00      0.00         3
   Very Poor       0.00      0.00      0.00        22

    accuracy                           0.19       183
   macro avg       0.04      0.20      0.06       183
weighted avg       0.04      0.19      0.06       183



## 4. First Linear Regression Model — Ishan (Week 4)

**Goal:** Train the first ordinary least-squares regression model for tomorrow’s PM2.5, using Deepanshu’s chronological split and training-fitted imputer. No random split or test-set fitting is performed here.

**Note:** The earlier `DummyRegressor(strategy="mean")` is a mean baseline, **not** the required persistence baseline. Abhishekk’s persistence comparison should be added separately. Atishey will conduct the formal model comparison.


In [8]:
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np

# Reuse the same features, target, and chronological split established above.
assert X_train_reg_imputed.index.equals(y_train_reg.index)
assert X_test_reg_imputed.index.equals(y_test_reg.index)
assert not X_train_reg_imputed.isna().any().any()
assert not X_test_reg_imputed.isna().any().any()
assert (X_train_reg_imputed.index.max() + pd.Timedelta(days=1)) < pd.Timestamp("2020-01-01")

linear_regression = LinearRegression()
linear_regression.fit(X_train_reg_imputed, y_train_reg)
linear_predictions = linear_regression.predict(X_test_reg_imputed)

print("Linear Regression trained successfully")
print("Training samples:", len(X_train_reg_imputed))
print("Testing samples:", len(X_test_reg_imputed))
print("Predictions generated:", len(linear_predictions))


Linear Regression trained successfully
Training samples: 1823
Testing samples: 183
Predictions generated: 183


### Preview next-day PM2.5 predictions

Each index date below is the **feature date**; the actual/predicted PM2.5 corresponds to the **following calendar day**. These are predictions, not evaluation scores.


In [9]:
linear_prediction_table = pd.DataFrame({
    "Feature Date": X_test_reg_imputed.index,
    "Prediction Date": X_test_reg_imputed.index + pd.Timedelta(days=1),
    "Actual PM2.5 Tomorrow": y_test_reg.to_numpy(),
    "Predicted PM2.5 Tomorrow": linear_predictions
})

assert len(linear_prediction_table) == len(y_test_reg)
assert np.isfinite(linear_predictions).all()
display(linear_prediction_table.head(10))


,Feature Date,Prediction Date,Actual PM2.5 Tomorrow,Predicted PM2.5 Tomorrow
0,2019-12-31,2020-01-01,372.14,218.688891
1,2020-01-01,2020-01-02,327.04,336.293044
2,2020-01-02,2020-01-03,228.90,298.768406
3,2020-01-03,2020-01-04,220.60,212.521188
4,2020-01-04,2020-01-05,170.21,207.115953
5,2020-01-05,2020-01-06,159.64,154.959226
6,2020-01-06,2020-01-07,120.31,147.594899
7,2020-01-07,2020-01-08,117.98,115.517425
8,2020-01-08,2020-01-09,89.26,113.390893
9,2020-01-09,2020-01-10,130.82,88.137423


### Week 4 handoff notes

- The fitted model is `linear_regression` and its test predictions are `linear_predictions`.
- The comparison table is `linear_prediction_table`; it can be used by Atishey for MAE, RMSE, R² and persistence comparison.
- **Data-quality review:** Check the upstream feature list for any columns derived from future information. AQI is a same-day aggregate and should be included only if available at prediction time. The next-day target columns are excluded upstream.
- The model is not yet evidence of improvement over persistence; evaluate on the unchanged test period before making that claim.
